In [1]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def check_overfitting(model, X_train, X_test, y_train, y_test, model_name="Model"):

    # 預測
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # 計算 R²
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    # MSE / RMSE
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    print(f"\n🔍 {model_name} Overfitting Check")
    print("--------------------------------------")
    print(f"Train R²: {train_r2:.3f}")
    print(f"Test  R²: {test_r2:.3f}")
    print(f"Train RMSE: {train_rmse:.3f}")
    print(f"Test  RMSE: {test_rmse:.3f}")

    # 判斷
    if train_r2 - test_r2 > 0.10 and train_r2 > 0.80:
        print("判定：Overfitting")
    elif test_r2 < 0.20 and train_r2 > test_r2:
        print("高度懷疑")
    else:
        print("未過擬合")

    return train_r2, test_r2


C:\Users\sara9\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error

# =========================
# 1.讀取與準備資料
# =========================
df = pd.read_csv('NBA_Normalized_Final.csv')

# 布林欄位處理
bool_cols = ['Is_Undrafted', 'Pos_PG', 'Pos_C', 'Pos_SG', 'Pos_SF', 'Pos_PF']
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.upper().map({"TRUE": 1, "FALSE": 0})

# 篩選資料 (三年級 WS 預測)
df_g = df[(df['season'] > 1) & (df['year_start'] > 1999)].copy()

feature_cols = [
    '2P%','2PA','3P','3P%','3PA','3PAr','AST','AST%','BLK','BLK%','BPM','DBPM',
    'DRB','DRB%','DWS','FG','FG%','FGA','FT','FT%','FTA','FTr',
    'OBPM','ORB','ORB%','OWS','PER','PF','PTS','Pos_C','Pos_PF','Pos_PG',
    'Pos_SF','Pos_SG','STL','STL%','TOV','TOV%','TRB','TRB%','TS%','USG%',
    'VORP','WS','WS/48','eFG%'
]
target_col = 'Year3_WS'

# 清洗目標值 
data = df_g.dropna(subset=[target_col]).copy()
for c in feature_cols:
    data[c] = pd.to_numeric(data[c], errors='coerce')
data[target_col] = pd.to_numeric(data[target_col], errors='coerce')
data = data.dropna(subset=[target_col]).copy()

X = data[feature_cols].copy()
y = data[target_col].copy()

# =========================
# 2.切分 train/test
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 3.建立 Pipeline 與 GridSearchCV 
# =========================
param_grid = {'pls__n_components': range(1, 21)}
cv_splits = min(5, len(X_train))

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), 
    ("scaler", StandardScaler()),                  
    ("pls", PLSRegression())
])

grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=cv_splits, 
    scoring='r2', 
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# 取出最佳模型
best_model = grid_search.best_estimator_
best_n_components = grid_search.best_params_['pls__n_components']
best_cv_r2 = grid_search.best_score_

# =========================
# 4.預測 
# =========================
y_train_pred = best_model.predict(X_train).ravel()
y_test_pred = best_model.predict(X_test).ravel()

# =========================
# 5.計算基本誤差與 R2
# =========================
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test  = np.sqrt(mean_squared_error(y_test, y_test_pred))

mae_test = mean_absolute_error(y_test, y_test_pred)
medae_test = median_absolute_error(y_test, y_test_pred)

# =========================
# 6.CV RMSE 
# =========================
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    best_model, X_train, y_train,
    scoring="neg_root_mean_squared_error",
    cv=cv, n_jobs=-1
)
cv_mean = -cv_scores.mean()
cv_std  = cv_scores.std(ddof=1)

# =========================
# 7.Train–Test Gap
# =========================
gap = rmse_test - rmse_train

# =========================
# 8.Spearman rank correlation
# =========================
def spearman_corr(y_true, y_pred):
    rt = np.argsort(np.argsort(y_true))
    rp = np.argsort(np.argsort(y_pred))
    return np.corrcoef(rt, rp)[0, 1]

spearman = spearman_corr(y_test, y_test_pred)

# =========================
# 9.輸出報告
# =========================
print("\n" + "="*45)
print("          PLS REGRESSION EVALUATION")
print("="*45)
print(f"Chosen n_components:      {best_n_components}")
print(f"Train R²:                 {train_r2:.6f}")
print(f"Best CV R² (train_5fold): {best_cv_r2:.6f}")
print(f"Test R²:                  {test_r2:.6f}")
print("-" * 45)
print(f"RMSE (test):              {rmse_test:.6f}")
print(f"MAE  (test):              {mae_test:.6f}")
print(f"MedianAE (test):          {medae_test:.6f}")
print(f"CV RMSE (train, 5-fold):  {cv_mean:.6f} ± {cv_std:.6f}")
print(f"Train–Test Gap:           {gap:.6f}")
print(f"Spearman rank corr (test):{spearman:.6f}")
print("="*45)


          PLS REGRESSION EVALUATION
Chosen n_components:      4
Train R²:                 0.514058
Best CV R² (train_5fold): 0.461369
Test R²:                  0.458480
---------------------------------------------
RMSE (test):              2.000880
MAE  (test):              1.606834
MedianAE (test):          1.376154
CV RMSE (train, 5-fold):  1.980190 ± 0.060883
Train–Test Gap:           0.085184
Spearman rank corr (test):0.634851


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score

# ---------------------------
# 1.切分資料
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------
# 2.Pipeline：標準化 + PLS
# ---------------------------
pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("pls", PLSRegression())
])

# ---------------------------
# 3.設定要搜尋的 n_components
# ---------------------------
param_grid = {
    "pls__n_components": list(range(1, 13))  
}
# ---------------------------
# 4.交叉驗證 + GridSearch
# ---------------------------
cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="r2",        
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid.fit(X_train, y_train)

print("Best n_components:", grid.best_params_["pls__n_components"])
print("Best CV R²:", grid.best_score_)

# ---------------------------
# 5.最佳模型在 test 上評估
# ---------------------------
best_model = grid.best_estimator_
y_test_pred = best_model.predict(X_test).ravel()
test_r2 = r2_score(y_test, y_test_pred)
print("Test R²:", test_r2)

# ---------------------------
# 6.把結果整理成表（方便你貼報告）
# ---------------------------
cv_results = pd.DataFrame(grid.cv_results_)
cv_results["n_components"] = cv_results["param_pls__n_components"].astype(int)

summary = cv_results[[
    "n_components",
    "mean_train_score",
    "std_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]].sort_values("n_components")

print("\nPLS tuning summary (first 10 rows):")
print(summary.head(10))

# ---------------------------
# 5.畫圖：R² vs n_components（
# ---------------------------
plt.figure(figsize=(8, 5))

x = summary["n_components"].values
cv_mean = summary["mean_test_score"].values
cv_std  = summary["std_test_score"].values

train_mean = summary["mean_train_score"].values
train_std  = summary["std_train_score"].values

# CV R²）
plt.errorbar(x, cv_mean, yerr=cv_std, capsize=3, marker="o", linestyle="-", label="CV R²")

# Train R²
plt.errorbar(x, train_mean, yerr=train_std, capsize=3, marker="s", linestyle="--", label="Train R²")

# 標示最佳點
best_k = grid.best_params_["pls__n_components"]
best_cv = grid.best_score_
plt.axvline(best_k, linestyle=":", linewidth=2)
plt.scatter([best_k], [best_cv], s=80)
plt.text(best_k, best_cv, f"  best k={best_k}", va="bottom")

plt.xlabel("n_components")
plt.ylabel("R²")
plt.title("PLS Tuning: R² vs n_components (Train vs CV)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

Fitting 5 folds for each of 12 candidates, totalling 60 fits


ValueError: 
All the 60 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
60 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\sara9\anaconda3\anaconda\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\sara9\anaconda3\anaconda\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\sara9\anaconda3\anaconda\Lib\site-packages\sklearn\pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\sara9\anaconda3\anaconda

In [7]:
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error
from scipy.stats import spearmanr

# -------------------------
# 0) 預處理：確保目標變數 y 沒有 NaN
# -------------------------
mask = ~np.isnan(y)
X_clean = X[mask]
y_clean = y[mask]

# -------------------------
# 1.切分訓練集與測試集 (80/20)
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, 
    test_size=0.2, 
    random_state=42
)

# -------------------------
# 2.建立 Pipeline 
# -------------------------
n_comp = n_components if 'n_components' in locals() else 2

pls_eval_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("pls", PLSRegression(n_components=n_comp))
])

# -------------------------
# 3.訓練與預測
# -------------------------
pls_eval_model.fit(X_train, y_train)

y_train_pred = pls_eval_model.predict(X_train).ravel()
y_test_pred  = pls_eval_model.predict(X_test).ravel()

# -------------------------
# 4.計算統計指標
# -------------------------
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test  = np.sqrt(mean_squared_error(y_test, y_test_pred))

mae_test   = mean_absolute_error(y_test, y_test_pred)
medae_test = median_absolute_error(y_test, y_test_pred)

# Spearman 相關係數
spearman, _ = spearmanr(y_test, y_test_pred)

# -------------------------
# 5.交叉驗證 (CV RMSE)
# -------------------------
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    pls_eval_model, 
    X_train, y_train, 
    scoring="neg_root_mean_squared_error", 
    cv=cv, 
    n_jobs=-1
)
cv_mean = -cv_scores.mean()
cv_std  = (-cv_scores).std(ddof=1)

# -------------------------
# 6.Train–Test Gap 
# -------------------------
gap = rmse_test - rmse_train

# -------------------------
# 7.輸出 結果
# -------------------------
print("\n===== PLS Regression Evaluation =====")
print(f"n_components: {n_comp}")
print(f"RMSE (test): {rmse_test:.6f}")
print(f"MAE  (test): {mae_test:.6f}")
print(f"MedianAE (test): {medae_test:.6f}")
print(f"CV RMSE (train, 5-fold): {cv_mean:.6f} ± {cv_std:.6f}")
print(f"Train–Test Gap (RMSE_test - RMSE_train): {gap:.6f}")
print(f"Spearman rank corr (test): {spearman:.6f}")


===== PLS Regression Evaluation =====
n_components: 4
RMSE (test): 2.000880
MAE  (test): 1.606834
MedianAE (test): 1.376154
CV RMSE (train, 5-fold): 1.980190 ± 0.060883
Train–Test Gap (RMSE_test - RMSE_train): 0.085184
Spearman rank corr (test): 0.636751
